In [ ]:
import os
import pandas as pd
import torch
from torch.utils.data import DataLoader, Dataset
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from tqdm.auto import tqdm
import re
import joblib

In [ ]:
# def replace_gaps(text):
#     if pd.isna(text): 
#         return text
    
#     text = re.sub(r'\.3(?:\s+\.3)+\.{3}(?:\s+\.{3})+\s+\.{3}(?:\s+\.{3})+', '<big_gap>', text)
#     text = re.sub(r'\.3(?:\s+\.3)+\.{3}(?:\s+\.{3})+', '<big_gap>', text)
#     text = re.sub(r'\.{3}(?:\s+\.{3})+', '<big_gap>', text)

#     text = re.sub(r'xx', '<gap>', text)
#     text = re.sub(r' x ', ' <gap> ', text)
#     text = re.sub(r'……', '<big_gap>', text)
#     text = re.sub(r'\.\.\.\.\.\.', '<big_gap>', text)
#     text = re.sub(r'…', '<big_gap>', text)
#     text = re.sub(r'\.\.\.', '<big_gap>', text)

#     return text
    

In [ ]:
def replace_gaps_back(text):
    if pd.isna(text):  
        return text
    
    text = re.sub(r'<gap>', 'x', text)
    text = re.sub(r'<big_gap>', '...', text)

    return text

In [ ]:
# MODEL_PATH = "/kaggle/input/dcp-train/byt5-base-akkadian-gap/"
MODEL1_PATH = "/kaggle/input/byt5-base-big-data2"
MODEL2_PATH = "/kaggle/input/byt5-akkadian-model"
MODEL3_PATH = "/kaggle/input/train-gap-all-2/byt5-base-akkadian_gap_setence2"

In [ ]:
import torch
import pandas as pd
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
import re  # 如果 replace_gaps 使用正则的话

# ================================================
# 配置参数
# ================================================
TEST_DATA_PATH = "/kaggle/input/deep-past-initiative-machine-translation/test.csv"
BATCH_SIZE = 8
MAX_LENGTH = 512
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# 模型路径（请替换成你实际的路径）
MODEL1_PATH = "/kaggle/input/byt5-base-big-data2"
MODEL2_PATH = "/kaggle/input/byt5-akkadian-model"
MODEL3_PATH = "/kaggle/input/train-gap-all-2/byt5-base-akkadian_gap_setence2"

# 性能比例（相对于 Model 2）
perf1 = 0.98   # Model 1 有 Model 2 的 0.98x 性能
perf2 = 1.00   # Model 2 为基准
perf3 = 0.45   # Model 3 的性能

# 计算权重（简单线性比例 + 归一化）
total = perf1 + perf2 + perf3
w1 = perf1 / total
w2 = perf2 / total
w3 = perf3 / total

print(f"权重分配:")
print(f"  w1 (Model1): {w1:.4f}")
print(f"  w2 (Model2): {w2:.4f}")
print(f"  w3 (Model3): {w3:.4f}")
print(f"  总和: {w1 + w2 + w3:.4f}\n")

# ================================================
# 加载模型
# ================================================
print("Loading Model 1...")
model1 = AutoModelForSeq2SeqLM.from_pretrained(MODEL1_PATH)
sd1 = model1.state_dict()

print("Loading Model 2...")
model2 = AutoModelForSeq2SeqLM.from_pretrained(MODEL2_PATH)
sd2 = model2.state_dict()

print("Loading Model 3...")
model3 = AutoModelForSeq2SeqLM.from_pretrained(MODEL3_PATH)
sd3 = model3.state_dict()

# ================================================
# 加权平均权重
# ================================================
print("Averaging weights (weighted ensemble)...")
final_sd = sd2.copy()  # 以 model2 为基础（权重最高时最稳定）

for key in final_sd:
    if key in sd1 and key in sd3:
        final_sd[key] = (
            w1 * sd1[key] +
            w2 * sd2[key] +
            w3 * sd3[key]
        )
    elif key in sd1:
        final_sd[key] = w1 * sd1[key] + (w2 + w3) * sd2[key]  # fallback
    elif key in sd3:
        final_sd[key] = w3 * sd3[key] + (w1 + w2) * sd2[key]
    # else: 保持 model2 原值

# 加载融合后的权重到 model2（或新建一个模型实例）
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL2_PATH)  # 结构以 model2 为准
model.load_state_dict(final_sd)
model = model.to(DEVICE)
model.eval()

print("Ensemble model loaded successfully!")

# ================================================
# Tokenizer（使用 Model 2 的 tokenizer，通常三个模型共享同一 tokenizer）
# ================================================
tokenizer = AutoTokenizer.from_pretrained(MODEL2_PATH)

# ================================================
# 读取测试集并应用 gap 替换
# ================================================
test_df = pd.read_csv(TEST_DATA_PATH)

# 你的 gap 替换函数（请确保已定义）
def replace_gaps(text):
    # 示例实现（根据你的实际需求修改）
    text = re.sub(r'\.{3,}|\.{2,}|x+x+|\bx\b|\s+x\s+', '<big_gap>', text)
    text = re.sub(r'\bx\b', '<gap>', text)
    return text

test_df['transliteration'] = test_df['transliteration'].apply(replace_gaps)


print("Data preparation completed.")
print(f"Test set size: {len(test_df)} rows")

In [ ]:
PREFIX = "translate Akkadian to English: "

class InferenceDataset(Dataset):
    def __init__(self, df, tokenizer):
        self.texts = df['transliteration'].astype(str).tolist()
        self.texts = [PREFIX + i for i in self.texts]
        self.tokenizer = tokenizer
        
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        text = self.texts[idx]
        inputs = self.tokenizer(
            text, 
            max_length=MAX_LENGTH, 
            padding="max_length", 
            truncation=True, 
            return_tensors="pt"
        )
        return {
            "input_ids": inputs["input_ids"].squeeze(0),
            "attention_mask": inputs["attention_mask"].squeeze(0)
        }

test_dataset = InferenceDataset(test_df, tokenizer)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print("Starting Inference...")
all_predictions = []

In [ ]:
import torch
from tqdm import tqdm

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = model.to(DEVICE)
model.eval()
model.float()

all_predictions = []

torch.set_grad_enabled(False)

with torch.inference_mode():
    for batch in tqdm(test_loader):
        input_ids = batch["input_ids"].to(DEVICE)              # int64, normal
        attention_mask = batch["attention_mask"].to(DEVICE)    # int64/bool, normal

        outputs = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_length=MAX_LENGTH,
            num_beams=4,
            early_stopping=True,
        )

        decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)
        all_predictions.extend([d.strip() for d in decoded])
        
        predictions = []
        for txt in all_predictions:
            predictions.append(txt)

In [ ]:
print(next(model.parameters()).dtype)

In [ ]:
def clean_repetitions(text):
    if not isinstance(text, str):
        return text

    text = re.sub(r'\b(\w+)(?:\s+\1\b)+', r'\1', text)
    
    for n in range(4, 1, -1): 
        pattern = r'\b((?:\w+\s+){' + str(n-1) + r'}\w+)(?:\s+\1\b)+'
        text = re.sub(pattern, r'\1', text)
        
    return text

In [ ]:
submission = pd.DataFrame({
    "id": test_df["id"],
    "translation": predictions
})
submission["translation"] = submission["translation"].apply(clean_repetitions)
submission["translation"] = submission["translation"].apply(lambda x: x if len(x) > 0 else "broken text")

    
submission.to_csv("submission.csv", index=False)
print("Submission file saved successfully!")
submission.head()